# **数理工学モデル化演習 岩永担当1回目 課題**
次のセルに学籍番号と氏名を記入してください。

学籍番号：202410178
氏名：今村隼人

以下の課題１、２を遂行してください。

## **課題１**
次の数理モデルとその実装について考えます。

### 数理モデル

> - リスト  
> $P$:製品のリスト  
> $M$:原料のリスト
> - 定数  
> $stock_{m}\ \ (m\in M)$:原料$m$の在庫量  
> $require_{p,m}\ \ (p\in P, m\in M)$:製品$p$を生産するのに必要な原料$m$の量  
> $gain_{p}\ \ (p\in P)$:製品$p$を生産した際の利得  
> - 変数  
> $x_p\ \ (p \in P)$:製品$p$の生産量
> - 制約式  
> $x_p \geq 0\ \ (p \in P)$:製品$p$の生産量は0以上  
> $\Sigma_{p\in P}require_{p,m} \cdot x_p \leq stock_m\ \ (m\in M)$:生産は在庫の範囲で行う
> - 目的関数（最大化）  
> $\Sigma_{p\in P}gain_p\cdot x_p$

### 数理モデルの実装

```python
import pandas as pd
import pulp

# データの取得
stock_df = pd.read_csv('stock.csv')
gain_df = pd.read_csv('gain.csv')
require_df = pd.read_csv('require.csv')

# 集合の定義
P = gain_df['p'].tolist()
M = stock_df['m'].tolist()

# 定数の定義
m2stock = {row.m:row.stock 
           for row in stock_df.itertuples()}
p2gain = {row.p:row.gain 
          for row in gain_df.itertuples()}
pm2require = {(row.p,row.m):row.require 
              for row in require_df.itertuples()}

# 最大化問題として数理モデルを定義
prob = pulp.LpProblem('LP', pulp.LpMaximize)

# 変数の定義
x = pulp.LpVariable.dicts('x', P, lowBound=0, cat='Continuous')

# 制約式の定義
for m in M:
    prob += pulp.lpSum([pm2require[p,m] * x[p] for p in P]) <= m2stock[m]

# 目的関数の定義    
prob += pulp.lpSum([p2gain[p] * x[p] for p in P])

# 求解
status = prob.solve()
print('Status:', pulp.LpStatus[status])

for p in P:
    print(p, x[p].value())
print('obj=', prob.objective.value())
```

上記のコードについて、次の（１）〜（１２）の各コードについて説明してください。講義内容を踏まえてできるだけ具体的に記述してください。
ただし、Web検索は利用してよいものとしますが、生成AIは利用しないでください。

（１）

```python
import pandas as pd
```

データ解析用ライブラリ`pandas`を読み込み，`pd`で参照できるようにしている

（２）

```python
import pulp
```

線形計画問題などの数理最適化問題を定式化・求解するためのライブラリ`pulp`を読み込んでいる

（３）

```python
stock_df = pd.read_csv('stock.csv')
```

`stock.csv`を`pandas`のデータフレームとして読み込み，その結果を`stock_df`に代入している

（４）

```python
P = gain_df['p'].tolist()
```

`gain_df`の`p`列を取り出し，製品の一覧を表すPythonのリストに変換している

（５）

```python
m2stock = {row.m:row.stock for row in stock_df.itertuples()}
```

`stock_df.itertuples()`で各行を順に取り出し，原料名`m`をキー，在庫量`stock`を値とする辞書を`m2stock`に代入している

（６）

```python
prob = pulp.LpProblem('LP', pulp.LpMaximize)
```

`pulp`の線形計画問題を作成し，目的関数を最大化する問題として`prob`に代入している

（７）

```python
x = pulp.LpVariable.dicts('x', P, lowBound=0, cat='Continuous')
```

各製品`p`に対応する変数`x[p]`をまとめて作成している。各変数は0以上連続変数で制約されている

（８）

```python
for m in M:
    prob += pulp.lpSum([pm2require[p,m] * x[p] for p in P]) <= m2stock[m]
```

各原料`m`についての制約式を1本ずつ追加している。左辺はその原料を使う全製品`p`の使用量`pm2require[p,m] * x[p]`の総和，右辺は在庫量`m2stock[m]`であり，総使用量が在庫を超えないことを表す

（９）

```python
prob += pulp.lpSum([p2gain[p] * x[p] for p in P])
```

全製品`p \in P`について，1単位あたりの利得`p2gain[p]`と生産量`x[p]`の積を合計した総利得を目的関数として`prob`に追加している

（１０）

```python
status = prob.solve()
```

`prob.solve()`を実行して定義した最適化問題を実際に求解し，求解結果を表すステータスコードを`status`に代入している

（１１）

```python
print('Status:', pulp.LpStatus[status])
```

`status`の数値コードを`pulp.LpStatus[status]`で`Optimal`や`Infeasible`のような文字列に変換し，`Status:`とあわせて標準出力に出力している

（１２）

```python
for p in P:
    print(p, x[p].value())
```

各製品`p`について順に，製品名`p`と最適解における生産量`x[p].value()`を標準出力に出力している

## **課題２**
（１）本講義を通じて学んだことを、400字以上で記述してください。ただし、生成AIを利用しないこと。

本講義を通して、数理最適化はただ計算して答えを出すための技術ではなく、現実の問題をどのように数理モデルに落とし込むかがとても大事だと学んだ。特に印象に残ったのは、現実世界とモデルの世界にはどうしてもギャップがあり、データや数理モデルはあくまで現実を近似したものにすぎないという点である。機械学習やこれまでのデータ分析でも似たことを感じていたが、どれだけ正確に数式を立ててプログラムを実装しても、モデル自体が現実をうまく捉えられていなければ、実際の意思決定には役立たないのだと改めて感じた。

また、数理最適化モデルは定数・変数・制約式・目的関数の4つの要素からできていて、それらを整理して書くことで問題を明確に表せることも学んだ。講義では、連立一次方程式、線形計画問題、整数計画問題をPuLPで実装したが、普段自分が書いているRustやTypeScriptのような手続き的なプログラムとは違って、変数どうしの関係を制約として表現する考え方はとても新鮮だった。さらに、集合を使って変数や制約を一般化したり、データとモデルを分けて扱ったりすることで、入力が変わっても柔軟に対応できる点は実践的で重要だと感じた。今後は最適化の課題に取り組むとき、ただ解法を当てはめるのではなく、何を変数にして、どの条件を制約にし、何を目的として最適化するのかを意識しながら、モデル化の段階から丁寧に考えていきたい。

（２）上記（１）で作成した文章をもとに、生成AIを用いて加筆・修正した回答を作成してください。

本講義を通じて、数理最適化とは単に計算によって最適な答えを求める技術ではなく、現実の課題をどのように抽象化し、数理モデルとして表現するかが本質であると学んだ。特に重要だと感じたのは、現実世界と数理モデルの間には必ず隔たりがあり、データやモデルはあくまで現実を近似的に記述したものにすぎないという点である。そのため、数式として整ったモデルを構築し、プログラム上で正しく解けたとしても、モデル化の前提や仮定が実情に合っていなければ、実際の意思決定に有効な結果にはならない。この視点は、これまで取り組んできた機械学習やデータ分析にも共通しており、モデルの精巧さだけでなく、問題設定そのものの妥当性を吟味する姿勢の重要性を改めて認識した。

また、数理最適化モデルが、定数・変数・制約式・目的関数という基本要素によって体系的に構成されることを学び、それぞれの役割を整理して記述することで複雑な問題を明確に扱えることを理解した。講義では、連立一次方程式、線形計画問題、整数計画問題をPuLPで実装したが、これは普段扱っている手続き型のプログラミングとは異なり、条件や目的を宣言的に記述する考え方に触れる機会となった。さらに、集合を用いて変数や制約を一般化し、データとモデルを分離して設計することで、入力が変化しても再利用しやすい柔軟な実装が可能になる点も印象的であった。今後は、最適化やアルゴリズム設計の課題に取り組む際、単に解法を当てはめるのではなく、何を意思決定変数とし、どの条件を制約として課し、何を評価指標として最適化するのかを意識しながら、モデル化の段階から丁寧に考えていきたい。

（３）上記（１）と（２）を比較し、違いや気づきがあれば述べてください。

（１）と（２）を比較すると、（２）は（１）の内容自体は大きく変わっていないものの、文章の構成や接続が整理され、主張と理由の関係がより明確になっていると感じた。特に、数理最適化の本質が「計算」ではなく「モデル化」にあるという中心的な考えが、（２）ではよりはっきり伝わる文章になっている。